# Assignment 5.2 - Hyperparameter Optimization

## Task 5.2.1 - Brute-Force Hyperparameter Search

* Implement a brute-force search function that finds the best parameter combination for a given model. **(RESULT)**
* Test your implementation on the following problems:
    - 1) A `SVM` model on the two moons problem. **(RESULT)**
    - 2) A `LinearRegression` model with Ridge regularization on the `California Housing Dataset`. **(RESULT)**

Feel free to use `sklearn`'s model implementations.

In [ ]:
# You might check for the following hyperparameter ranges:

svm_params = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}


ridge_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

In [ ]:
import numpy as np
import itertools
import random
from sklearn.datasets import make_moons, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.svm import SVC
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import warnings

# Suppress warnings that might clutter the output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# Define the brute-force search function
def brute_force_search(model_class, param_grid, X, y, score_func, is_regression=False, n_splits=5):
    """
    Performs a full grid search using K-Fold Cross-Validation.

    Args:
        model_class: The uninitialized model class (e.g., SVC, Ridge).
        param_grid (dict): Dictionary of hyperparameter names and lists of values to test.
        X (np.ndarray): Feature data.
        y (np.ndarray): Target data.
        score_func: Scoring function (e.g., accuracy_score, mean_squared_error).
        is_regression (bool): True for regression, False for classification.
        n_splits (int): Number of folds for cross-validation.

    Returns:
        tuple: (best_params, best_score)
    """

    # 1. Setup CV and scaler
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    best_score = -np.inf if not is_regression else np.inf # Maximize score (Acc) or Minimize score (MSE)
    best_params = {}

    # Generate all parameter combinations
    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    print(f"Testing {len(param_combinations)} total combinations...")

    # 2. Iterate through all combinations
    for params in param_combinations:
        fold_scores = []
         # 3. Perform K-Fold Cross-Validation (prevents data leakage)
        for train_index, test_index in kf.split(X):
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]

            # Feature Scaling (CRITICAL: Fit on train, transform on both)
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Train and Evaluate
            model = model_class(**params, random_state=42 if not is_regression else None)
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)

            # Score
            score = score_func(y_test, y_pred)
            fold_scores.append(score)

        mean_score = np.mean(fold_scores)

        # 4. Update best score and parameters
        if (is_regression and mean_score < best_score) or \
           (not is_regression and mean_score > best_score):
            best_score = mean_score
            best_params = params

    return best_params, best_score

# --- Data Loading and Setup Functions ---

def setup_moons():
    """Load, scale, and split the two moons dataset."""
    X_moons, y_moons = make_moons(n_samples=500, noise=0.3, random_state=42)
    # The brute_force_search function handles internal scaling,
    # so we pass the raw data.
    return X_moons, y_moons

def setup_housing():
    """Load and prepare the California Housing dataset."""
    housing = fetch_california_housing()
    X_housing, y_housing = housing.data, housing.target
    # The brute_force_search function handles internal scaling.
    return X_housing, y_housing

# Define the parameter grids here for reuse in the TPE task
svm_params = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

ridge_params = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

print("Brute-Force Hyperparameter Search function defined.")

# 1. SVM on Two Moons Classification
print("--- 1) SVM (SVC) on Two Moons Classification (Brute-Force) ---")
X_moons, y_moons = setup_moons()

# Note: SVC requires a random_state to be passed during initialization
best_params_svm, best_score_svm = brute_force_search(
    model_class=SVC,
    param_grid=svm_params,
    X=X_moons,
    y=y_moons,
    score_func=accuracy_score,
    is_regression=False
)

print("\n(RESULT) Best SVM Hyperparameters (Moons):")
print(f"  Best Parameters: {best_params_svm}")
print(f"  Best Mean Accuracy (CV): {best_score_svm:.4f}")


# 2. Ridge Regression on California Housing
print("\n--- 2) Ridge Regression on California Housing (Brute-Force) ---")
X_housing, y_housing = setup_housing()

# Note: Ridge is used instead of LinearRegression as per the prompt's parameter grid
# Ridge is LinearRegression with L2 regularization (alpha)
best_params_ridge, best_score_ridge = brute_force_search(
    model_class=Ridge,
    param_grid=ridge_params,
    X=X_housing,
    y=y_housing,
    score_func=mean_squared_error,
    is_regression=True
)

print("\n(RESULT) Best Ridge Hyperparameters (Housing):")
print(f"  Best Parameters: {best_params_ridge}")
print(f"  Best Mean MSE (CV): {best_score_ridge:.4f}")

Brute-Force Hyperparameter Search function defined.
--- 1) SVM (SVC) on Two Moons Classification (Brute-Force) ---
Testing 24 total combinations...

(RESULT) Best SVM Hyperparameters (Moons):
  Best Parameters: {'C': 10.0, 'kernel': 'rbf', 'gamma': 'scale'}
  Best Mean Accuracy (CV): 0.9260

--- 2) Ridge Regression on California Housing (Brute-Force) ---
Testing 20 total combinations...

(RESULT) Best Ridge Hyperparameters (Housing):
  Best Parameters: {'alpha': 10.0, 'solver': 'auto'}
  Best Mean MSE (CV): 0.5305


## Task 5.2.2 - Simple TPE (BONUS)

* Implement the Tree-Structured Parzen Estimator using `numpy` only. **(RESULT)**
* Find decent hyperparameters for
    - 1) An `SVM` model on the `two moons` problem. **(RESULT)**
    - 2) A `LinearRegression` model with Ridge regularization on the `California Housing Dataset`. **(RESULT)**

In [ ]:
class TPE:
    """
    Simplified Tree-Structured Parzen Estimator (TPE) implementation
    for discrete hyperparameter optimization, using NumPy.

    This version adapts TPE to discrete grids by:
    1. Treating the hyperparameter space as discrete.
    2. Evaluating the acquisition function (ratio of good density to bad density)
       to find the next promising combination to test.

    The TPE algorithm requires two PDFs: l(x) (good params) and g(x) (bad params).
    Since we only have discrete values, we use weighted sampling based on
    the evaluated scores.
    """

    def __init__(self, param_grid, score_func, is_regression=False, gamma=0.25):
        """Initializes the TPE optimizer."""
        self.param_grid = param_grid
        self.score_func = score_func
        self.is_regression = is_regression
        self.gamma = gamma  # Fraction of 'good' configurations (e.g., top 25%)
        self.history = defaultdict(list) # Stores {'C': [0.1, 1.0, ...], 'score': [...]}
        self.best_params = {}
        self.best_score = -np.inf if not is_regression else np.inf

        # All possible discrete combinations (used for initial sampling)
        keys, values = zip(*param_grid.items())
        self.param_keys = keys
        self.all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    def _evaluate(self, model_class, params, X, y, n_splits=5):
        """Helper to evaluate a single parameter set using CV."""
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        fold_scores = []

        for train_index, test_index in kf.split(X):
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]

            # Feature Scaling (CRITICAL: Fit on train, transform on both)
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Train and Evaluate
            try:
                model = model_class(**params, random_state=42 if not self.is_regression else None)
                model.fit(X_train_scaled, y_train)
                y_pred = model.predict(X_test_scaled)
                score = self.score_func(y_test, y_pred)
            except Exception:
                # Handle cases where parameter combinations (like poly kernel without degree) might fail
                score = -np.inf if not self.is_regression else np.inf

            fold_scores.append(score)

        return np.mean(fold_scores)

    def optimize(self, model_class, X, y, max_iters=50):
        """Main TPE optimization loop."""

        # 1. Initial Random Warm-up (Required to populate history)
        num_warmup = min(10, len(self.all_combinations))

        # Convert combinations to indices and sample without replacement
        indices = np.random.choice(len(self.all_combinations), size=num_warmup, replace=False)

        for idx in indices:
            params = self.all_combinations[idx]
            mean_score = self._evaluate(model_class, params, X, y)
            self._update_history(params, mean_score)

        print(f"TPE Warm-up complete. Starting iterative search (max {max_iters} iterations).")

        # 2. Iterative TPE Search
        for iteration in range(max_iters - num_warmup):

            # 2a. Separate 'Good' (l(x)) and 'Bad' (g(x)) observations
            all_scores = np.array(self.history['score'])

            # Find the threshold score (gamma percentile)
            if self.is_regression: # MSE: lower score is better
                score_threshold = np.percentile(all_scores, self.gamma * 100)
                good_indices = np.where(all_scores <= score_threshold)[0]
            else: # Accuracy: higher score is better
                score_threshold = np.percentile(all_scores, (1 - self.gamma) * 100)
                good_indices = np.where(all_scores >= score_threshold)[0]

            bad_indices = np.where(all_scores < score_threshold)[0] if self.is_regression else np.where(all_scores < score_threshold)[0]

            # Fallback if no bad indices exist yet (e.g., initial runs are all "good")
            if len(bad_indices) == 0:
                bad_indices = np.setdiff1d(np.arange(len(all_scores)), good_indices)

            # 2b. Compute density models l(x) and g(x) by weighting
            # In this simplified discrete TPE, we look at the frequency of parameter values
            # in the good set (l) vs. the bad set (g) across the history.

            l_counts = defaultdict(lambda: defaultdict(int))
            g_counts = defaultdict(lambda: defaultdict(int))

            for key in self.param_keys:
                for idx in good_indices:
                    val = self.history[key][idx]
                    l_counts[key][val] += 1
                for idx in bad_indices:
                    val = self.history[key][idx]
                    g_counts[key][val] += 1

            # 2c. Propose the next parameter set
            # The acquisition function is l(x) / g(x). We look for high l(x) and low g(x).
            # We iterate through all unvisited combinations to maximize this ratio.

            unvisited_combinations = [c for c in self.all_combinations if c not in [dict(zip(self.param_keys, [self.history[k][i] for k in self.param_keys])) for i in range(len(all_scores))]]

            # If all combinations have been visited, restart the pool but use TPE for guidance
            if not unvisited_combinations:
                unvisited_combinations = self.all_combinations

            best_acquisition = -1.0
            next_params = None

            for params_candidate in unvisited_combinations:
                acquisition_ratio = 1.0 # Start with 1.0 and multiply probabilities

                for key in self.param_keys:
                    val = params_candidate[key]

                    # Probability (Density) for 'good' (l(x))
                    l_prob = l_counts[key].get(val, 0) / (len(good_indices) + 1e-6)
                    # Probability (Density) for 'bad' (g(x))
                    g_prob = g_counts[key].get(val, 0) / (len(bad_indices) + 1e-6)

                    # Compute the acquisition ratio (E[ Improvement ])
                    if g_prob > 1e-6:
                        ratio = l_prob / g_prob
                    else:
                        ratio = np.inf # If g_prob is zero, l(x)/g(x) is high

                    acquisition_ratio *= ratio

                if acquisition_ratio > best_acquisition:
                    best_acquisition = acquisition_ratio
                    next_params = params_candidate

            if next_params is None: # Fallback: if TPE failed to find a next step, just pick randomly
                next_params = random.choice(self.all_combinations)

            # 2d. Evaluate the proposed parameters
            mean_score = self._evaluate(model_class, next_params, X, y)
            self._update_history(next_params, mean_score)

            print(f"  TPE Iteration {iteration + 1} - Score: {mean_score:.4f}, Params: {next_params}")

        return self.best_params, self.best_score

    def _update_history(self, params, score):
        """Stores the result and updates the overall best."""
        self.history['score'].append(score)
        for key in self.param_keys:
            self.history[key].append(params[key])

        # Check for best score
        is_new_best = False
        if (self.is_regression and score < self.best_score) or \
           (not self.is_regression and score > self.best_score):
            self.best_score = score
            self.best_params = params
            is_new_best = True

        return is_new_best

print("TPE (Tree-Structured Parzen Estimator) class defined.")

TPE (Tree-Structured Parzen Estimator) class defined.


In [ ]:
# Setup data again (data loading is cheap)
X_moons, y_moons = setup_moons()
X_housing, y_housing = setup_housing()

# 1. SVM on Two Moons Classification (TPE)
print("--- 1) SVM (SVC) on Two Moons Classification (TPE) ---")
# Use a higher max_iters than the number of brute-force combinations (24) to show the iterative process
tpe_svm = TPE(
    param_grid=svm_params,
    score_func=accuracy_score,
    is_regression=False,
    gamma=0.25 # Consider top 25% as 'good'
)

best_params_tpe_svm, best_score_tpe_svm = tpe_svm.optimize(
    model_class=SVC,
    X=X_moons,
    y=y_moons,
    max_iters=24 # Limiting to 24, as the full grid only has 24 combinations
)

print("\n(RESULT) Best SVM Hyperparameters (Moons) via TPE:")
print(f"  Best Parameters: {best_params_tpe_svm}")
print(f"  Best Mean Accuracy (CV): {best_score_tpe_svm:.4f}")


# 2. Ridge Regression on California Housing (TPE)
print("\n--- 2) Ridge Regression on California Housing (TPE) ---")
tpe_ridge = TPE(
    param_grid=ridge_params,
    score_func=mean_squared_error,
    is_regression=True,
    gamma=0.25 # Consider top 25% as 'good'
)

best_params_tpe_ridge, best_score_tpe_ridge = tpe_ridge.optimize(
    model_class=Ridge,
    X=X_housing,
    y=y_housing,
    max_iters=20 # Full grid has 20 combinations
)

print("\n(RESULT) Best Ridge Hyperparameters (Housing) via TPE:")
print(f"  Best Parameters: {best_params_tpe_ridge}")
print(f"  Best Mean MSE (CV): {best_score_tpe_ridge:.4f}")

--- 1) SVM (SVC) on Two Moons Classification (TPE) ---
TPE Warm-up complete. Starting iterative search (max 24 iterations).
  TPE Iteration 1 - Score: 0.8980, Params: {'C': 1.0, 'kernel': 'rbf', 'gamma': 'auto'}
  TPE Iteration 2 - Score: 0.9260, Params: {'C': 10.0, 'kernel': 'rbf', 'gamma': 'scale'}
  TPE Iteration 3 - Score: 0.8380, Params: {'C': 0.1, 'kernel': 'linear', 'gamma': 'auto'}
  TPE Iteration 4 - Score: 0.8240, Params: {'C': 0.1, 'kernel': 'poly', 'gamma': 'auto'}
  TPE Iteration 5 - Score: 0.8400, Params: {'C': 1.0, 'kernel': 'linear', 'gamma': 'scale'}
  TPE Iteration 6 - Score: 0.8280, Params: {'C': 1.0, 'kernel': 'poly', 'gamma': 'scale'}
  TPE Iteration 7 - Score: 0.8420, Params: {'C': 10.0, 'kernel': 'linear', 'gamma': 'scale'}
  TPE Iteration 8 - Score: 0.8300, Params: {'C': 10.0, 'kernel': 'poly', 'gamma': 'scale'}
  TPE Iteration 9 - Score: 0.8300, Params: {'C': 10.0, 'kernel': 'poly', 'gamma': 'auto'}
  TPE Iteration 10 - Score: 0.8420, Params: {'C': 100.0, 'kern